# Egyptian ID — Complete Unified Pipeline

**Full Flow:**
1. **Load references** — real + fake ID images (cached between runs)
2. **Select FRONT** of test ID
3. **Auto-rotate** — correct orientation via rembg + green-strip detection
4. **Verify** — dual-reference gate (reject fakes)
5. **Dual OCR** — run both Full System + YOLO pipelines independently
6. **Smart Merge** — pick best result per field using scoring + bonuses
7. **Select BACK** of test ID (optional)
8. **Back OCR** — extract expiration date
9. **Final Result** — combined output with source tracking

## Cell 1 — Imports & Configuration

In [1]:
%load_ext autoreload
%autoreload 2

import cv2
import numpy as np
import re
import os
import sys
import string
import pytesseract
from tkinter import *
from tkinter import filedialog
from PIL import Image
from rembg import remove as rembg_remove
from easyocr import easyocr
from datetime import datetime
from ultralytics import YOLO
from skimage.metrics import structural_similarity as ssim

# Shared modules
from ocr_helpers import choose_address, _clean_name, _extract_birthdate_from_id, arabic_to_english_numbers
from ocr_enhancements import validate_national_id
import rotation_app
import result_logger

# ─── Tesseract paths ─────────────────────────────────────────────────────────
pytesseract.pytesseract.tesseract_cmd = r'D:\ocr\tesseract\tesseract.exe'
os.environ['TESSDATA_PREFIX'] = r'D:\ocr\tessdata'

# ─── Shared constants ─────────────────────────────────────────────────────────
_ARABIC_TO_WESTERN = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')
_WESTERN_TO_ARABIC = str.maketrans('0123456789', '٠١٢٣٤٥٦٧٨٩')

# ─── Verification configuration ──────────────────────────────────────────────
OUTPUT_SIZE         = (640, 400)
PASSING_SCORE       = 50.0
FAKE_MARGIN         = 0
MAX_REFERENCES      = 3
MAX_FAKE_REFERENCES = 10
REFS_SAVE_PATH      = r"D:\ocr\national_id_ocr\references_cache.npz"

ZONES = [
    (0.00, 0.22, 0.25, 1.00, 0.08, "header"),
    (0.25, 0.75, 0.30, 0.80, 0.30, "gold_center"),
    (0.80, 1.00, 0.00, 1.00, 0.30, "bottom_strip"),
    (0.00, 0.22, 0.00, 0.25, 0.20, "top_left"),
    (0.25, 0.90, 0.45, 1.00, 0.12, "guilloche_bg"),
]
ZONE_PRESENCE_THRESHOLD = 0.25

# ─── Init models (done once) ─────────────────────────────────────────────────
YOLO_MODEL_PATH = r"D:\ocr\national_id_ocr\dual_pipeline_package\detect_odjects.pt"
print("Loading YOLO model...")
yolo_model = YOLO(YOLO_MODEL_PATH)
print("Loading EasyOCR...")
reader = easyocr.Reader(['ar', 'en'], gpu=False)
print("Models ready.")

C:\Users\Omar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading rembg model …
rembg ready.
Loading YOLO model for rotation face-check...


Using CPU. Note: This module is much faster with a GPU.


Loading YOLO model...
Loading EasyOCR...
Models ready.


## Cell 2 — Verification Functions

Dual-reference verification gate: compares the test ID against real and fake reference images using SSIM + color histogram across 5 zones.

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
#  VERIFICATION FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def _find_card_corners(img_bgr):
    orig_h, orig_w = img_bgr.shape[:2]
    scale = min(1.0, 1000.0 / orig_w)
    work  = cv2.resize(img_bgr, (int(orig_w * scale), int(orig_h * scale)))
    try:
        pil_img   = Image.fromarray(cv2.cvtColor(work, cv2.COLOR_BGR2RGB))
        pil_out   = rembg_remove(pil_img)
        rgba      = np.array(pil_out)
        mask      = (rgba[:, :, 3] > 10).astype(np.uint8) * 255
        card_work = cv2.cvtColor(rgba[:, :, :3], cv2.COLOR_RGB2BGR)
    except Exception:
        card_work = work
        mask = np.ones(work.shape[:2], dtype=np.uint8) * 255
    masked  = cv2.bitwise_and(card_work, card_work, mask=mask)
    gray    = cv2.cvtColor(masked, cv2.COLOR_BGR2GRAY)
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray    = clahe.apply(gray)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges   = cv2.Canny(blurred, 30, 120)
    kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    edges   = cv2.dilate(edges, kernel, iterations=2)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, scale
    contours  = sorted(contours, key=cv2.contourArea, reverse=True)
    work_area = work.shape[0] * work.shape[1]
    for cnt in contours[:5]:
        if cv2.contourArea(cnt) < work_area * 0.10:
            continue
        peri   = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)
        if len(approx) == 4:
            return approx, scale
    x, y, w, h = cv2.boundingRect(contours[0])
    rect = np.array([[[x,y]],[[x+w,y]],[[x+w,y+h]],[[x,y+h]]], dtype=np.int32)
    return rect, scale


def _order_points(pts):
    pts  = pts.reshape(4, 2).astype(np.float32)
    rect = np.zeros((4, 2), dtype=np.float32)
    s    = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]; rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]; rect[3] = pts[np.argmax(diff)]
    return rect


def _extract_card(original_bgr, corners_scaled, scale):
    pts   = _order_points(corners_scaled / scale)
    w1    = np.linalg.norm(pts[1]-pts[0]); w2 = np.linalg.norm(pts[2]-pts[3])
    h1    = np.linalg.norm(pts[3]-pts[0]); h2 = np.linalg.norm(pts[2]-pts[1])
    max_w = int(max(w1, w2)); max_h = int(max(h1, h2))
    if max_h > max_w: max_w, max_h = max_h, max_w
    dst   = np.array([[0,0],[max_w,0],[max_w,max_h],[0,max_h]], dtype=np.float32)
    M     = cv2.getPerspectiveTransform(pts, dst)
    return cv2.warpPerspective(original_bgr, M, (max_w, max_h))


def _remove_shadow(img_bgr):
    result_channels = []
    kh = max(10, img_bgr.shape[0]//5); kw = max(10, img_bgr.shape[1]//5)
    kh = kh if kh%2==1 else kh+1; kw = kw if kw%2==1 else kw+1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kw, kh))
    for ch in cv2.split(img_bgr):
        bg   = cv2.morphologyEx(ch, cv2.MORPH_CLOSE, kernel)
        norm = (ch.astype(np.float32) / (bg.astype(np.float32)+1e-6)) * 200.0
        result_channels.append(np.clip(norm, 0, 255).astype(np.uint8))
    return cv2.merge(result_channels)


def _preprocess_card(img_bgr, label=""):
    corners, scale = _find_card_corners(img_bgr)
    card = _extract_card(img_bgr, corners.astype(np.float32), scale) if corners is not None else img_bgr.copy()
    if card.shape[0] > card.shape[1]:
        card = cv2.rotate(card, cv2.ROTATE_90_CLOCKWISE)
    card = _remove_shadow(card)
    lab     = cv2.cvtColor(card, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l       = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8,8)).apply(l)
    card    = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
    return cv2.resize(card, OUTPUT_SIZE)


def _crop_zone(img, y1p, y2p, x1p, x2p):
    h, w = img.shape[:2]
    return img[int(h*y1p):int(h*y2p), int(w*x1p):int(w*x2p)]


def _zone_ssim(a, b):
    ga = cv2.cvtColor(a, cv2.COLOR_BGR2GRAY)
    gb = cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)
    ms = min(ga.shape + gb.shape)
    win = min(7, ms if ms%2==1 else ms-1)
    if win < 3: return 0.0
    sc, _ = ssim(ga, gb, full=True, win_size=win)
    return max(0.0, float(sc))


def _zone_color(a, b):
    ha = cv2.cvtColor(a, cv2.COLOR_BGR2HSV)
    hb = cv2.cvtColor(b, cv2.COLOR_BGR2HSV)
    scores = []
    for ch in range(3):
        h1 = cv2.calcHist([ha],[ch],None,[32],[0,256]); cv2.normalize(h1,h1)
        h2 = cv2.calcHist([hb],[ch],None,[32],[0,256]); cv2.normalize(h2,h2)
        scores.append(max(0.0, float(cv2.compareHist(h1,h2,cv2.HISTCMP_CORREL))))
    return float(np.mean(scores))


def _compare_zone(test, ref, zone_def):
    y1p, y2p, x1p, x2p, w, name = zone_def
    zt = _crop_zone(test, y1p, y2p, x1p, x2p)
    zr = _crop_zone(ref,  y1p, y2p, x1p, x2p)
    if zt.size == 0 or zr.size == 0: return 0.0, False
    raw      = _zone_ssim(zt, zr)*0.55 + _zone_color(zt, zr)*0.45
    baseline = 1.0*0.55 + _zone_color(zr, zr)*0.45
    score    = min((raw / max(baseline, 0.01))*100.0, 100.0)
    return score, score >= ZONE_PRESENCE_THRESHOLD*100.0


def _score_against_set(test, ref_set, label="REF"):
    best_score = 0.0; best_name = ""
    for i, (ref_img, ref_name) in enumerate(ref_set):
        tw = ts = 0.0; rejects = []
        for zd in ZONES:
            y1p,y2p,x1p,x2p,weight,name = zd
            sc, present = _compare_zone(test, ref_img, zd)
            _, ref_pres = _compare_zone(ref_img, ref_img, zd)
            if ref_pres and not present: rejects.append(name)
            ts += sc * weight; tw += weight
        final = ts/tw if tw > 0 else 0.0
        final = max(0.0, final - len(rejects)*15.0)
        if final > best_score: best_score = final; best_name = ref_name
    return round(best_score, 1), best_name


def verify_image(test_img, real_refs, fake_refs):
    """Dual-reference classifier. Returns (is_authentic, real_score, fake_score)."""
    real_score, real_best = _score_against_set(test_img, real_refs, "REAL")
    fake_score, fake_best = _score_against_set(test_img, fake_refs, "FAKE") if fake_refs else (0.0, "—")

    print(f"\n{' -'*55}")
    print(f"  [ VERIFICATION RESULT ]")
    print(f"  Real score : {real_score:.1f}  (vs {real_best})")
    if fake_refs:
        print(f"  Fake score : {fake_score:.1f}  (vs {fake_best})")
        print(f"  Margin     : {real_score:.1f} − {fake_score:.1f} = {real_score-fake_score:.1f}  (need > {FAKE_MARGIN})")
    print(f"  Threshold  : {PASSING_SCORE}")
    print(f"{' -'*55}")

    if fake_refs and fake_score >= real_score - FAKE_MARGIN:
        print(f"  ❌ IMAGE REJECTED  (Δ={real_score-fake_score:.1f})")
        return False, real_score, fake_score
    elif real_score >= PASSING_SCORE:
        print(f"  ✅ VERIFIED — Proceeding to OCR...")
        return True, real_score, fake_score
    else:
        print(f"  ❌ IMAGE REJECTED — Score {real_score:.1f} below threshold {PASSING_SCORE}")
        return False, real_score, fake_score


# ─── Reference Management ─────────────────────────────────────────────────────

def _load_refs(paths, label):
    refs = []
    for path in paths:
        img = cv2.imread(path)
        if img is None: continue
        name = os.path.basename(path)
        print(f"  Loading {label}: {name}")
        refs.append((_preprocess_card(img), name))
    return refs


def save_references(real_refs, fake_refs, save_path=REFS_SAVE_PATH):
    arrays = {}
    for i, (img, name) in enumerate(real_refs):
        arrays[f"real_img_{i}"]  = img
        arrays[f"real_name_{i}"] = np.array([name])
    arrays["real_count"] = np.array([len(real_refs)])
    for i, (img, name) in enumerate(fake_refs):
        arrays[f"fake_img_{i}"]  = img
        arrays[f"fake_name_{i}"] = np.array([name])
    arrays["fake_count"] = np.array([len(fake_refs)])
    np.savez_compressed(save_path, **arrays)
    print(f"  ✓ References saved to {save_path}")


def load_saved_references(save_path=REFS_SAVE_PATH):
    if not os.path.exists(save_path):
        return None, None
    data_np = np.load(save_path, allow_pickle=True)
    real_count = int(data_np["real_count"][0])
    fake_count = int(data_np["fake_count"][0])
    real_refs  = [(data_np[f"real_img_{i}"], str(data_np[f"real_name_{i}"][0]))
                  for i in range(real_count)]
    fake_refs  = [(data_np[f"fake_img_{i}"], str(data_np[f"fake_name_{i}"][0]))
                  for i in range(fake_count)]
    return real_refs, fake_refs


def _pick_files(title, multiple=True):
    w = Tk(); w.withdraw(); w.attributes('-topmost', True)
    if multiple:
        paths = filedialog.askopenfilenames(parent=w, title=title,
                    filetypes=[("Images","*.jpg *.jpeg *.png *.bmp")])
    else:
        p = filedialog.askopenfilename(parent=w, title=title,
                    filetypes=[("Images","*.jpg *.jpeg *.png *.bmp")])
        paths = [p] if p else []
    w.destroy()
    return list(paths)

## Cell 3 — Pipeline 1: YOLO Pipeline

Uses YOLO object detection to crop each field (firstName, lastName, address, nid), then runs OCR on each crop independently.

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
#  PIPELINE 1: YOLO-based OCR
# ═══════════════════════════════════════════════════════════════════════════════

def _extract_digits_yolo(text):
    text = str(text).translate(_ARABIC_TO_WESTERN)
    return re.sub(r'\D', '', text)

def _clean_arabic_yolo(text):
    text = re.sub(r'[^؀-ۿ0-9\s\-]', ' ', text or "")
    return ' '.join(text.split())


def run_yolo_pipeline(file_path, debug=False):
    data = {
        "first name": "0",
        "seconed name": "0",
        "address": "0",
        "id": "0",
        "birthdate": "0",
        "error": 0
    }
    try:
        img = cv2.imread(file_path)
        
        results = yolo_model.predict(img, conf=0.25, verbose=False)
        crops = {}
        
        for box in results[0].boxes:
            class_id = int(box.cls[0].item())
            class_name = yolo_model.names[class_id]
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            
            crop = img[y1:y2, x1:x2]
            
            if class_name not in crops or (crop.shape[0]*crop.shape[1] > crops[class_name].shape[0]*crops[class_name].shape[1]):
                crops[class_name] = crop
        
        if 'firstName' in crops:
            fn_text = pytesseract.image_to_string(crops['firstName'], lang='ara', config='--psm 6 --oem 3')
            data['first name'] = _clean_arabic_yolo(fn_text)
            
        if 'lastName' in crops:
            ln_text = pytesseract.image_to_string(crops['lastName'], lang='ara', config='--psm 6 --oem 3')
            data['seconed name'] = _clean_arabic_yolo(ln_text)
            
        if 'address' in crops:
            addr_img = crops['address']
            addr_img = cv2.GaussianBlur(addr_img, (3,3), 0)
            addr_img = cv2.convertScaleAbs(addr_img, alpha=1.3, beta=10)
            addr_text = pytesseract.image_to_string(addr_img, lang='ara', config='--psm 6 --oem 3')
            data['address'] = ' '.join(_clean_arabic_yolo(addr_text).split())
            
        if 'nid' in crops:
            nid_gray = cv2.cvtColor(crops['nid'], cv2.COLOR_BGR2GRAY)
            gauss = cv2.GaussianBlur(nid_gray, (7,7), 0)
            unsharp = cv2.addWeighted(nid_gray, 2, gauss, -1, 0)
            
            nid_results = reader.readtext(unsharp, detail=0, text_threshold=0.25, width_ths=0.8, low_text=0.01)
            
            if len(nid_results) == 1:
                raw_id = nid_results[0]
            elif len(nid_results) == 0:
                raw_id = "0"
            else:
                raw_id = max(nid_results, key=len)
                
            raw_id = _extract_digits_yolo(raw_id)
            
            if len(raw_id) == 14 and raw_id.endswith('2'):
                reversed_id = raw_id[::-1]
                if reversed_id.startswith('2') or reversed_id.startswith('3'):
                    raw_id = reversed_id
                    
            validated_id = validate_national_id(raw_id)
            data['id'] = validated_id
            
            if len(validated_id) == 14 and validated_id[0] in ('2', '3'):
                century = "19" if validated_id[0] == '2' else "20"
                year = century + validated_id[1:3]
                month = validated_id[3:5]
                day = validated_id[5:7]
                data['birthdate'] = f"{year}-{month}-{day}"
    except Exception as e:
        print(f"YOLO pipeline error: {e}")
        data['error'] = 1

    if data['birthdate'] == "0":
        data['error'] = 1

    return data

## Cell 4 — Pipeline 2: Full System Pipeline

Uses fixed-ratio pixel cuts based on image dimensions. Dual OCR engine (Tesseract + EasyOCR) with smart address merging.

In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
#  PIPELINE 2: Full System OCR (fixed-ratio cuts)
# ═══════════════════════════════════════════════════════════════════════════════

def run_full_system_pipeline(file_path):
    data = {
        "first name": "0", "seconed name": "0", "address": "0",
        "id": "0", "birthdate": "0", "error": 0
    }
    try:
        img = cv2.imread(file_path)
        cropped = img
        w,h,c = cropped.shape
        o = int(w/2); i = int(h/2.5); n = int(h/6)
        cr          = cropped[n-13:i+15, o:]
        cropped_img = cropped[i+8:, o+10:]

        cr_height   = cr.shape[0]; split_point = int(cr_height * 0.52)
        names_region   = cr[0:split_point, :]
        address_region = cr[split_point:, :] 
        address_region = cv2.GaussianBlur(address_region, (3,3), 0)
        address_region = cv2.convertScaleAbs(address_region, alpha=1.3, beta=10)

        text_names   = pytesseract.image_to_string(names_region, lang='ara', config='--psm 11 --oem 3')
        splited_names = text_names.split('\n')

        arabic_digits = ["٠","١","٢","٣","٤","٥","٦","٧","٨","٩"]
        pun = set(string.punctuation)

        s         = easyocr.Reader(['ar','ar'], gpu=False)
        d_names   = s.readtext(names_region, detail=0, text_threshold=0.18, width_ths=0.9, low_text=0.17)
        d_address = s.readtext(address_region, detail=0, text_threshold=0.15, width_ths=0.7, low_text=0.15, paragraph=True)

        state = 0
        if len(text_names.split('\n')) == 4:
            state = 1
            data["first name"]   = splited_names[0] if len(splited_names)>0 else "0"
            data["seconed name"] = splited_names[2] if len(splited_names)>2 else "0"
            address_easyocr      = ' '.join(d_address) if d_address else ""
            data["address"]      = choose_address("", address_easyocr)

            imgs = cv2.cvtColor(cropped_img, cv2.COLOR_BGR2GRAY)
            gauss = cv2.GaussianBlur(imgs, (7,7), 0)
            unsharp = cv2.addWeighted(imgs, 2, gauss, -1, 0)
            o2 = s.readtext(unsharp, detail=0, text_threshold=0.27, width_ths=0.8, low_text=0.008)
            if len(o2)==1:   data["id"] = o2[0]
            elif len(o2)==0: data["id"] = "0"
            else:            data["id"] = max(o2, key=len)
        else:
            state = 2
            imgs  = cv2.cvtColor(cropped_img, cv2.COLOR_BGR2GRAY)
            imgs  = cv2.medianBlur(imgs, 3)
            gauss = cv2.GaussianBlur(imgs, (5,5), 0)
            unsharp = cv2.addWeighted(imgs, 2.2, gauss, -1.2, 0)
            unsharp = cv2.convertScaleAbs(unsharp, alpha=1.4, beta=5)
            d = d_names
            address_easyocr = ' '.join(d_address) if d_address else ""
            for tok in d:
                if tok in arabic_digits: break
                my_data    = ','.join(d)
                split_list = my_data.split(',')
                data["first name"]   = split_list[0] if split_list else "0"
                data["seconed name"] = ','.join(split_list[1:]).replace(","," ").strip() if len(split_list)>1 else "0"
                data["address"]      = choose_address("", address_easyocr).replace("[","").replace("]","").replace("'","")
            o2 = s.readtext(unsharp, detail=0, text_threshold=0.25, width_ths=0.7, low_text=0.01, paragraph=False)
            if o2 is None or d is None: data["error"] = "1"
            elif len(o2)==1:   data["id"] = o2[0]
            elif len(o2)==0:   data["id"] = "0"
            else:              data["id"] = max(o2, key=len)

        if len(str(data["id"])) < 20: data["error"] = "1"
        for c in list(data["first name"]):
            if c in arabic_digits or c in pun:
                data["first name"] = data["first name"].replace(c, "")
        for c in list(data["seconed name"]):
            if c in arabic_digits or c in pun:
                data["seconed name"] = data["seconed name"].replace(c, "")
        if len(data["seconed name"]) <= len(data["first name"]):
            data["first name"], data["seconed name"] = data["seconed name"], data["first name"]

        first_parts  = [p for p in (data["first name"] or "").split() if p]
        second_parts = [p for p in (data["seconed name"] or "").split() if p]
        if len(first_parts) > 3:  data["error"] = "1"
        if len(second_parts) <= 1: data["error"] = "1"

        ar = list('ابتثجحخدذرزسشصضطظعغفقكلمنهوي')
        for c in list(str(data["id"])):
            if c in ar or c in pun:
                data["id"] = str(data["id"]).replace(c, "")

        arabic_string = str(data["id"])
        matches = re.findall(r'[٠-٩]+', arabic_string)
        matches.reverse()
        concatenated = ''.join(matches)
        if concatenated:
            data["id"] = int(concatenated.translate(_ARABIC_TO_WESTERN))

        data["birthdate"] = _extract_birthdate_from_id(data["id"])
        if data["birthdate"] == "0": data["error"] = "1"
        
        data["id"] = validate_national_id(str(data["id"]))
        
    except Exception as e:
        print("Full System pipeline error:", e)
        data['error'] = 1

    return data

## Cell 5 — Smart Output Merger

Scores each field from both pipelines and picks the best result. Includes bonus points for valid dates, address keywords, and cross-checking NID governorate against address.

In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
#  SMART MERGE — Best-of-two field picker
# ═══════════════════════════════════════════════════════════════════════════════

def _digits_only(s):
    return re.sub(r'\D', '', (s or "").translate(_ARABIC_TO_WESTERN))

def score_name(s):
    if not s or s == "0": return -1
    if any(ch.isdigit() for ch in s): return -1
    s_clean = re.sub(r'[ً-ٟ]', '', s)
    arabic_chars = len(re.findall(r'[ء-يپچژگ]', s_clean))
    spaces = s_clean.count(' ')
    total_len = len(s_clean)
    noise = total_len - (arabic_chars + spaces)
    if noise > 0:
        return arabic_chars - (noise * 5)
    return arabic_chars

def score_address(s):
    if not s or s == "0": return -1
    words = s.split()
    if len(words) < 2: return -1
    s_clean = re.sub(r'[ً-ٟ]', '', s)
    arabic_chars = len(re.findall(r'[ء-يپچژگ]', s_clean))
    digits = len(re.findall(r'[0-9٠-٩]', s_clean))
    spaces = s_clean.count(' ')
    total_len = len(s_clean)
    valid_punct = len(re.findall(r'[\-ـ]', s_clean))
    noise = total_len - (arabic_chars + digits + spaces + valid_punct)
    if noise > 0:
        return (arabic_chars + digits) - (noise * 5)
    return arabic_chars + digits

def score_id(s):
    s_str = str(s)
    digits = _digits_only(s_str)
    if len(digits) != 14:
        return -1
    noise = len(s_str) - 14
    if noise > 0:
        return 14 - (noise * 5)
    return 14

import datetime

def bonus_id_date(s):
    s_str = _digits_only(str(s))
    if len(s_str) != 14:
        return 0
    try:
        century = "19" if s_str[0] == '2' else "20"
        year = int(century + s_str[1:3])
        month = int(s_str[3:5])
        day = int(s_str[5:7])
        datetime.date(year, month, day)
        return 5
    except ValueError:
        return 0

def bonus_address_keywords(s):
    if not s or s == "0": return 0
    keywords = ['مركز', 'قسم', 'شارع', 'محافظة', 'مدينة', 'قرية', 'كفر', 'عزبة', 'بلوك', 'مجاورة']
    s_clean = re.sub(r'[ً-ٟ]', '', s)
    bonus = 0
    for kw in keywords:
        if kw in s_clean:
            bonus += 1
    return bonus

GOVERNORATES = {
    "01": "القاهرة", "02": "الاسكندرية", "03": "بورسعيد", "04": "السويس",
    "11": "دمياط", "12": "الدقهلية", "13": "الشرقية", "14": "القليوبية",
    "15": "كفر الشيخ", "16": "الغربية", "17": "المنوفية", "18": "البحيرة",
    "19": "الاسماعيلية", "21": "الجيزة", "22": "بني سويف", "23": "الفيوم",
    "24": "المنيا", "25": "اسيوط", "26": "سوهاج", "27": "قنا",
    "28": "اسوان", "29": "الاقصر", "31": "البحر الاحمر", "32": "الوادي الجديد",
    "33": "مطروح", "34": "شمال سيناء", "35": "جنوب سيناء"
}

def bonus_cross_check(id_str, addr_str):
    if not id_str or not addr_str or id_str == "0" or addr_str == "0": return 0
    id_str = _digits_only(str(id_str))
    if len(id_str) != 14: return 0
    gov_code = id_str[7:9]
    if gov_code in GOVERNORATES:
        gov_name = GOVERNORATES[gov_code]
        addr_clean = re.sub(r'[ً-ٟ]', '', addr_str)
        addr_norm = addr_clean.replace("ة", "ه").replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
        gov_norm = gov_name.replace("ة", "ه").replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
        if gov_norm in addr_norm:
            return 10
    return 0


def smart_merge(full_data, yolo_data):
    final = {}
    sources = {}

    fid = str(full_data.get('id', '0'))
    yid = str(yolo_data.get('id', '0'))
    fa = str(full_data.get('address', '0'))
    ya = str(yolo_data.get('address', '0'))

    # Calculate bonuses
    f_cross_bonus = bonus_cross_check(fid, fa)
    y_cross_bonus = bonus_cross_check(yid, ya)

    f_id_bonus = bonus_id_date(fid) + f_cross_bonus
    y_id_bonus = bonus_id_date(yid) + y_cross_bonus

    f_addr_bonus = bonus_address_keywords(fa) + f_cross_bonus
    y_addr_bonus = bonus_address_keywords(ya) + y_cross_bonus

    # 1. ID
    fs_score_id = score_id(fid)
    if fs_score_id > 0: fs_score_id += f_id_bonus
    
    ys_score_id = score_id(yid)
    if ys_score_id > 0: ys_score_id += y_id_bonus
    
    if fs_score_id > ys_score_id:
        final['id'] = fid; sources['id'] = "full_system (better score)"
    elif ys_score_id > fs_score_id:
        final['id'] = yid; sources['id'] = "yolo (better score)"
    elif fs_score_id > 0 and ys_score_id > 0:
        final['id'] = fid; sources['id'] = "full_system (tie, both valid)"
    else:
        final['id'] = fid if len(fid) >= len(yid) else yid
        sources['id'] = "none (unvalidated)"

    # 2. Names
    for key in ['first name', 'seconed name']:
        fv = str(full_data.get(key, '0'))
        yv = str(yolo_data.get(key, '0'))
        fs_score = score_name(fv)
        ys_score = score_name(yv)
        
        if fs_score > ys_score:
            final[key] = fv; sources[key] = "full_system (better score)"
        elif ys_score > fs_score:
            final[key] = yv; sources[key] = "yolo (better score)"
        elif fs_score > 0 and ys_score > 0:
            final[key] = fv; sources[key] = "full_system (tie, both valid)"
        else:
            final[key] = fv if len(fv) >= len(yv) else yv
            sources[key] = "none (unvalidated)"

    # 3. Address
    fs_score_addr = score_address(fa)
    if fs_score_addr > 0: fs_score_addr += f_addr_bonus
    
    ys_score_addr = score_address(ya)
    if ys_score_addr > 0: ys_score_addr += y_addr_bonus
    
    if fs_score_addr > ys_score_addr:
        final['address'] = fa; sources['address'] = "full_system (better score)"
    elif ys_score_addr > fs_score_addr:
        final['address'] = ya; sources['address'] = "yolo (better score)"
    elif fs_score_addr > 0 and ys_score_addr > 0:
        final['address'] = fa; sources['address'] = "full_system (tie, both valid)"
    else:
        final['address'] = fa if len(fa) >= len(ya) else ya
        sources['address'] = "none (unvalidated)"

    # Ensure second name is actually longer/different from first name
    if final["seconed name"] != "0" and len(final["seconed name"]) <= len(final["first name"]):
        final["first name"], final["seconed name"] = final["seconed name"], final["first name"]

    # Finalize
    final['birthdate'] = _extract_birthdate_from_id(final['id']) if score_id(final['id']) > 0 else "0"
    final['error'] = 0 if score_id(final['id']) > 0 and final['birthdate'] != "0" else 1
    final['_sources'] = sources

    return final

## Cell 6 — Back-of-ID OCR

Extracts the expiration date from the back of the Egyptian national ID.

In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
#  BACK-ID OCR  (expiration date)
# ═══════════════════════════════════════════════════════════════════════════════

def run_back_ocr(file_path):
    """Extract expiration date from the BACK of the ID."""
    try:
        input_image  = Image.open(file_path)
        output_image = rembg_remove(input_image)
        img_array    = np.array(output_image)

        if len(img_array.shape)==3 and img_array.shape[2]==4:
            img_proc = cv2.cvtColor(img_array, cv2.COLOR_RGBA2BGRA)
        else:
            img_proc = img_array

        if len(img_proc.shape)==3 and img_proc.shape[2]==4:
            alpha  = img_proc[:,:,3]
            coords = cv2.findNonZero(alpha)
            if coords is not None:
                x,y,w,h = cv2.boundingRect(coords)
                img_proc = img_proc[y:y+h, x:x+w]

        h, w = img_proc.shape[:2]
        start_y = int(h*0.45); end_y = int(h*0.60)
        cropped = img_proc if start_y >= h else img_proc[start_y:end_y, :]
        if cropped.size == 0: return None
        cropped = cropped[:, :int(cropped.shape[1]*0.55)]
        if cropped.size == 0: return None

        gray   = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY) if len(cropped.shape)==3 else cropped
        gauss  = cv2.GaussianBlur(gray, (7,7), 0)
        sharp  = cv2.addWeighted(gray, 2, gauss, -1, 0)
        gauss2 = cv2.GaussianBlur(sharp, (7,7), 0)
        final  = cv2.addWeighted(sharp, 2, gauss2, -1, 0)

        back_reader = easyocr.Reader(['ar','ar'], gpu=False)
        results = back_reader.readtext(final, detail=0, text_threshold=0.27,
                                  width_ths=0.8, low_text=0.008, allowlist='٠١٢٣٤٥٦٧٨٩')
        print(f"Back OCR results: {results}")

        for text in results:
            text = ''.join(re.findall(r'[٠-٩]', text.strip()))
            if not text or len(text) < 6: continue
            text = arabic_to_english_numbers(text)
            if int(text[0]) < 2: text = text[1:]
            year      = text[:4]
            month_raw = text[5:7] if len(text)>6 else text[4:6]
            if int(month_raw) > 12: month_raw = month_raw[1]
            month = month_raw.zfill(2)
            day_raw = text[7:] if len(text)>7 else text[6:]
            if len(day_raw) > 2: day_raw = day_raw[1:]
            day = day_raw.zfill(2)
            return f"{year}/{month}/{day}"

        return None
    except Exception as e:
        print(f"Back OCR Error: {e}")
        return None

## Cell 7 — Main Execution

Complete flow: Load refs → Select front → Rotate → Verify → Dual OCR → Smart Merge → Back OCR → Final result.

In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
#  MAIN EXECUTION
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'═'*60}")
print("  EGYPTIAN ID — COMPLETE UNIFIED PIPELINE")
print(f"  Verification → Rotation → Dual OCR → Smart Merge → Back OCR")
print(f"{'═'*60}")

# ── STEP 1-2: Load references ────────────────────────────────────────────
real_refs = fake_refs = None
if os.path.exists(REFS_SAVE_PATH):
    real_refs, fake_refs = load_saved_references()
    if real_refs is not None:
        print(f"\n  Loaded from cache: {len(real_refs)} real + {len(fake_refs)} fake refs.")

if real_refs is None:
    print("\n  STEP 1 — Select 1-3 REAL reference ID images")
    real_paths = _pick_files("Select REAL reference images", multiple=True)
    if not real_paths:
        raise SystemExit("No real references selected.")
    real_refs = _load_refs(real_paths[:MAX_REFERENCES], "REAL")
    if not real_refs:
        raise SystemExit("No valid real references.")
    print("\n  STEP 2 — Select FAKE references (Cancel to skip)")
    fake_paths = _pick_files("Select FAKE reference images (optional)", multiple=True)
    fake_refs  = _load_refs(fake_paths[:MAX_FAKE_REFERENCES], "FAKE") if fake_paths else []
    save_references(real_refs, fake_refs)
    print("  References saved — will auto-load next run.")

print(f"  References ready: {len(real_refs)} real + {len(fake_refs)} fake.")

# ── STEP 3: Select FRONT image ───────────────────────────────────────────
print(f"\n  STEP 3 — Select the FRONT of the ID")
front_paths = _pick_files("Select FRONT of ID image", multiple=False)
if not front_paths:
    raise SystemExit("No image selected.")
front_path = front_paths[0]
print(f"  Image: {os.path.basename(front_path)}")

# ── STEP 4: Auto-rotate ───────────────────────────────────────────────
print(f"\n{'─'*60}")
print("  STEP 4 — Auto-rotating to correct orientation…")
print(f"{'─'*60}")
front_img, rotation_applied, rotation_note = rotation_app.auto_rotate_id_from_path(front_path)
norm_path = "temp_shared_normalized.png"
cv2.imwrite(norm_path, front_img)
print(f"  Rotation: {rotation_applied}deg")

# ── STEP 5: Verification ──────────────────────────────────────────────
print(f"\n{'─'*60}")
print("  STEP 5 — Verifying against references…")
print(f"{'─'*60}")
front_proc  = _preprocess_card(front_img)
is_auth, real_sc, fake_sc = verify_image(front_proc, real_refs, fake_refs)

if not is_auth:
    print(f"\n{'═'*60}")
    print("  ❌ ACCESS DENIED — FAKE or SUSPICIOUS image.")
    print("  OCR will NOT run.")
    print(f"{'═'*60}")
    raise SystemExit("Image rejected by verification.")

# ── STEP 6: Dual OCR + Smart Merge ──────────────────────────────────────
print(f"\n{'─'*60}")
print("  STEP 6 — Running Dual OCR Pipelines…")
print(f"{'─'*60}")

print("\n  [1/2] Full System Pipeline...")
full_sys_res = run_full_system_pipeline(norm_path)
full_sys_res['rotation'] = f"{rotation_applied}deg"

print("  [2/2] YOLO Pipeline...")
yolo_res = run_yolo_pipeline(norm_path)
yolo_res['rotation'] = f"{rotation_applied}deg"

print("\n  Merging outputs...")
final_result = smart_merge(full_sys_res, yolo_res)
final_result['rotation'] = f"{rotation_applied}deg"

# ── STEP 7: Back OCR (optional) ───────────────────────────────────────
print(f"\n{'─'*60}")
print("  STEP 7 — Select the BACK of the ID (for expiry date)")
print(f"{'─'*60}")
back_paths = _pick_files("Select BACK of ID image (Cancel to skip)", multiple=False)
expiration_date = None
if back_paths:
    print("  Running back-ID OCR…")
    expiration_date = run_back_ocr(back_paths[0])
    if expiration_date:
        print(f"  Expiration date: {expiration_date}")
    else:
        print("  Could not extract expiry date.")
else:
    print("  No back image — skipping expiry date.")

final_result['Expiration date'] = expiration_date

# ── Log the result ──────────────────────────────────────────────────
result_logger.log_result("unified_pipeline", front_path, final_result)

# ── Print all results ───────────────────────────────────────────────
print(f"\n{'═'*60}")
print("  FULL SYSTEM PIPELINE RESULT")
print(f"{'═'*60}")
for k, v in full_sys_res.items():
    print(f"  {k:<15}: {v}")

print(f"\n{'═'*60}")
print("  YOLO PIPELINE RESULT")
print(f"{'═'*60}")
for k, v in yolo_res.items():
    print(f"  {k:<15}: {v}")

print(f"\n{'═'*60}")
print("  ⭐ FINAL SMART MERGED RESULT")
print(f"{'═'*60}")
for k, v in final_result.items():
    if k != "_sources":
        print(f"  {k:<20}: {v}")

print(f"\n{'═'*60}")
print("  SOURCE TRACKING")
print(f"{'═'*60}")
for k, v in final_result["_sources"].items():
    print(f"  {k:<15}: {v}")

print(f"\n  Verification: {'✅ PASSED' if is_auth else '❌ FAILED'} (real={real_sc:.1f}, fake={fake_sc:.1f})")
print(f"{'═'*60}")


════════════════════════════════════════════════════════════
  EGYPTIAN ID — COMPLETE UNIFIED PIPELINE
  Verification → Rotation → Dual OCR → Smart Merge → Back OCR
════════════════════════════════════════════════════════════

  Loaded from cache: 3 real + 2 fake refs.
  References ready: 3 real + 2 fake.

  STEP 3 — Select the FRONT of the ID
  Image: tmg.png

────────────────────────────────────────────────────────────
  STEP 4 — Auto-rotating to correct orientation…
────────────────────────────────────────────────────────────
  Rotation: 0deg

────────────────────────────────────────────────────────────
  STEP 5 — Verifying against references…
────────────────────────────────────────────────────────────

 - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
  [ VERIFICATION RESULT ]
  Real score : 51.9  (vs WhatsApp Image 2026-02-16 at 5.12.29 PM.jpeg)
  Fake score : 47.6  (vs WhatsApp Image 2026-02-22 at 11.13.43 PM.jpg)
  M

Using CPU. Note: This module is much faster with a GPU.


  [2/2] YOLO Pipeline...

  Merging outputs...

────────────────────────────────────────────────────────────
  STEP 7 — Select the BACK of the ID (for expiry date)
────────────────────────────────────────────────────────────
  No back image — skipping expiry date.
[Logger] Saved unified_pipeline result to ocr_results.jsonl

════════════════════════════════════════════════════════════
  FULL SYSTEM PIPELINE RESULT
════════════════════════════════════════════════════════════
  first name     : محمد
  seconed name   : عبدالمنعم احمد عبده عزالدين
  address        : ٧٨ ش السويسرى ب مدينة نصرأول - القاهره
  id             : 29706100104091
  birthdate      : 1997-06-10
  error          : 0
  rotation       : 0deg

════════════════════════════════════════════════════════════
  YOLO PIPELINE RESULT
════════════════════════════════════════════════════════════
  first name     : محمد
  seconed name   : 
  address        : د ش السويسر مدينة لقاهره
  id             : 27960011004019
  birthdate     